# WallGuard 完整版：牆面危害 AI 辨識

拍一張牆面照片，判斷是**裂縫、壁癌還是鏽蝕**，標出缺損範圍、估計面積，再產生白話的處理建議。

```
照片 ─▶ ResNet50V2 判斷種類 ─┐
     └▶ U-Net 標出缺損範圍 ──┼─▶ 面積估計 ─▶ GPT 處理建議
```

專案開發時試過兩種做法：

- **初版**（`1_resnet_classification.ipynb`）：ResNet50V2 替整張照片分類，知道是哪一種牆損，但看不出位置與大小。
- **最終版**（`2_unet_segmentation.ipynb`）：U-Net 影像分割，標出範圍並換算面積，再串接 GPT 給建議。

最終版的完整程式已經找不到，這份筆記本在 2026 年把兩者合併成一條流程，並修正原始筆記本的已知問題：

| 問題（原始筆記本） | 這份的做法 |
|---|---|
| ResNet 用訓練資料評估準確率 | 分層切出 20% 驗證集，兩個模型共用同一份切分 |
| U-Net 的訓練／驗證集沒有打散，驗證集只有壁癌 | 打散後分層切分，三類都在驗證集 |
| U-Net 遮罩用固定門檻，沒有用到照片的類別，標籤只分明暗 | 依類別找出比周圍亮（壁癌）或暗（裂縫、鏽蝕）的區域，標成「缺損／背景」；裂縫只保留夠長的線條 |
| 遮罩存成 JPEG，壓縮雜訊變成假的類別 | 遮罩只放在記憶體，不經過 JPEG |
| 多類別 softmax 卻用 binary_crossentropy | 改成單一通道 sigmoid，損失函數用 binary cross-entropy＋Dice，並計算 IoU |
| 上傳的 RGB 照片用 BGR 轉灰階 | 一律用 RGB 轉換 |

> **限制：** U-Net 的遮罩是用門檻自動產生的近似標註，不是人工標註；面積以「每個像素 0.5 mm」粗估，沒有依拍攝距離校正。

### 執行前
1. **執行階段 → 變更執行階段類型 → T4 GPU**。
2. 想要 AI 建議的話，在左側欄 🔑 **Secrets** 新增 `OPENAI_API_KEY` 並開啟筆記本存取權；沒有設定也能跑，只是略過建議。
3. 從第一格依序執行。完整訓練在 T4 上約 20～30 分鐘；想先確認流程，把 `QUICK_TEST` 設成 `True`。

## 0. 設定

In [ ]:
QUICK_TEST = False   # True：每類只取 20 張、各訓練 1 個 epoch，只用來確認流程能跑通

CATEGORIES = ["crack", "efflorescence", "concrete"]                       # 資料夾名稱
LABELS_ZH = {"crack": "裂縫", "efflorescence": "壁癌", "concrete": "鏽蝕"}  # 顯示用的名稱
# 產生近似遮罩的規則：缺損比周圍亮（壁癌）還是暗（裂縫、鏽蝕）、比較的範圍（像素）、
# 只取反差最強的前幾 %、最短長度（裂縫只保留夠長的線條）
MASK_RULES = {
    "crack":         dict(bright=False, kernel=21, percentile=93, min_length=30),
    "efflorescence": dict(bright=True,  kernel=31, percentile=85, min_length=0),
    "concrete":      dict(bright=False, kernel=25, percentile=90, min_length=0),
}

DATASET_URL = "https://github.com/Yunyuhu/dtaidataset/raw/main/dataset.zip"
IMG_SIZE = 224
VAL_RATIO = 0.2
SEED = 42

RESNET_EPOCHS, RESNET_BATCH = 8, 8     # 沿用初版設定
UNET_EPOCHS, UNET_BATCH = 10, 4        # 最終版受限於 GPU 記憶體，batch size 4

PIXEL_MM = 0.5          # 假設每個像素 0.5 mm × 0.5 mm（沒有依拍攝距離校正）
MIN_CONFIDENCE = 0.5    # 種類的信心低於這個值，就顯示「無法確定」
OPENAI_MODEL = "gpt-4o-mini"   # 請依 OpenAI 目前提供的模型調整

if QUICK_TEST:
    RESNET_EPOCHS = UNET_EPOCHS = 1

## 1. 安裝與匯入套件

In [ ]:
!pip install -q gradio openai

In [ ]:
import os, random, zipfile, urllib.request
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50V2
from tensorflow.keras.applications.resnet_v2 import preprocess_input
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import gradio as gr
from openai import OpenAI

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
WORK_DIR = "/content" if os.path.isdir("/content") else os.path.abspath("wallguard_work")
os.makedirs(WORK_DIR, exist_ok=True)
print("TensorFlow", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "沒有偵測到 GPU，訓練會非常慢")

In [ ]:
# OpenAI 金鑰：優先讀 Colab Secrets，其次讀環境變數；都沒有就略過 AI 建議
api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
except Exception:
    api_key = os.environ.get("OPENAI_API_KEY")
client = OpenAI(api_key=api_key) if api_key else None
print("AI 建議：", "已啟用" if client else "未設定 OPENAI_API_KEY，將略過")

## 2. 下載資料並切分

3,000 張室內牆面照片，三類各 1,000 張，來自 Roboflow Universe 的公開資料集。兩個模型共用同一份訓練／驗證切分，驗證集的照片兩個模型都沒看過。

In [ ]:
zip_path = os.path.join(WORK_DIR, "dataset.zip")
data_dir = os.path.join(WORK_DIR, "dataset")
if not os.path.exists(zip_path):
    print("下載資料集（約 190 MB）…")
    urllib.request.urlretrieve(DATASET_URL, zip_path)
if not os.path.isdir(data_dir):
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(data_dir)

paths, labels = [], []
for k, cat in enumerate(CATEGORIES):
    folder = os.path.join(data_dir, cat)
    if not os.path.isdir(folder):
        raise FileNotFoundError(f"資料集裡找不到 {cat}/ 資料夾，請確認 DATASET_URL 是否正確")
    names = sorted(f for f in os.listdir(folder)
                   if f.lower().endswith((".jpg", ".jpeg")) and not f.startswith("._"))
    if QUICK_TEST:
        names = names[:20]
    paths += [os.path.join(folder, f) for f in names]
    labels += [k] * len(names)
paths, labels = np.array(paths), np.array(labels)

train_p, val_p, train_y, val_y = train_test_split(
    paths, labels, test_size=VAL_RATIO, stratify=labels, random_state=SEED)
for name, y in (("訓練", train_y), ("驗證", val_y)):
    print(name, len(y), "張：", {LABELS_ZH[c]: int((y == k).sum()) for k, c in enumerate(CATEGORIES)})

In [ ]:
def read_rgb(path, size=IMG_SIZE):
    """讀成 RGB、縮放成 size×size 的 uint8 陣列。"""
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f"讀不到圖片：{path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)

train_rgb = np.stack([read_rgb(p) for p in train_p])
val_rgb = np.stack([read_rgb(p) for p in val_p])
print("訓練", train_rgb.shape, "驗證", val_rgb.shape)

fig, axes = plt.subplots(1, len(CATEGORIES), figsize=(9, 3))
for ax, (k, cat) in zip(axes, enumerate(CATEGORIES)):
    ax.imshow(train_rgb[np.where(train_y == k)[0][0]]); ax.set_title(cat); ax.axis("off")
plt.show()

## 3. 模型一：ResNet50V2 判斷種類（初版做法）

沿用初版的遷移學習：預訓練的 ResNet50V2 去掉頂層、保留權重不訓練，只在最後接一層三類的 softmax。差別是這次用驗證集評估。

In [ ]:
def make_cls_ds(images, y, training):
    ds = tf.data.Dataset.from_tensor_slices((images, tf.one_hot(y, len(CATEGORIES))))
    if training:
        ds = ds.shuffle(len(images), seed=SEED)
    ds = ds.map(lambda x, t: (preprocess_input(tf.cast(x, tf.float32)), t),
                num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(RESNET_BATCH).prefetch(tf.data.AUTOTUNE)

backbone = ResNet50V2(include_top=False, pooling="avg", weights="imagenet",
                      input_shape=(IMG_SIZE, IMG_SIZE, 3))
backbone.trainable = False
cls_model = keras.Sequential([backbone, layers.Dense(len(CATEGORIES), activation="softmax")])
cls_model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

cls_history = cls_model.fit(make_cls_ds(train_rgb, train_y, True),
                            validation_data=make_cls_ds(val_rgb, val_y, False),
                            epochs=RESNET_EPOCHS)

In [ ]:
val_prob = cls_model.predict(make_cls_ds(val_rgb, val_y, False), verbose=0)
val_pred = val_prob.argmax(axis=1)
print(f"ResNet50V2 驗證集準確率：{(val_pred == val_y).mean():.2%}")
names = [LABELS_ZH[c] for c in CATEGORIES]
print(classification_report(val_y, val_pred, labels=list(range(len(CATEGORIES))),
                            target_names=CATEGORIES, digits=3, zero_division=0))
print("混淆矩陣（列：實際，欄：預測；順序：", "、".join(names), "）")
print(confusion_matrix(val_y, val_pred, labels=list(range(len(CATEGORIES)))))

## 4. 模型二：U-Net 標出缺損範圍（最終版做法，修正遮罩）

U-Net 架構沿用最終版，遮罩改成「缺損＝1、背景＝0」：

- 用形態學的 **top-hat** 找出比周圍亮的區域（壁癌），**black-hat** 找出比周圍暗的區域（裂縫、鏽蝕）。
- 只取反差最強的一部分，再去掉零碎的小點；裂縫只保留夠長的線條。

種類交給 ResNet50V2 判斷，U-Net 只負責範圍。缺損通常只占照片一小部分，所以損失函數用 binary cross-entropy 加上 Dice，避免模型全部猜背景。

> 這仍是自動產生的近似標註。例如裂縫資料夾裡混有沒有裂縫的牆面照片，遮罩會標到一些紋理。

In [ ]:
def to_gray(rgb):
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)

def make_mask(rgb, category):
    """依類別產生近似遮罩：缺損＝1、背景＝0。"""
    rule = MASK_RULES[category]
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (rule["kernel"], rule["kernel"]))
    op = cv2.MORPH_TOPHAT if rule["bright"] else cv2.MORPH_BLACKHAT
    response = cv2.GaussianBlur(cv2.morphologyEx(to_gray(rgb), op, kernel), (3, 3), 0)
    mask = (response >= max(np.percentile(response, rule["percentile"]), 12)).astype(np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))   # 接上斷掉的線段
    _, comp, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    longest = np.maximum(stats[:, cv2.CC_STAT_WIDTH], stats[:, cv2.CC_STAT_HEIGHT])
    keep = (stats[:, cv2.CC_STAT_AREA] >= 40) & (longest >= rule["min_length"])
    keep[0] = False                                                              # 0 是背景
    return keep[comp].astype(np.uint8)

train_gray = np.stack([to_gray(x) for x in train_rgb])
val_gray = np.stack([to_gray(x) for x in val_rgb])
train_mask = np.stack([make_mask(x, CATEGORIES[k]) for x, k in zip(train_rgb, train_y)])
val_mask = np.stack([make_mask(x, CATEGORIES[k]) for x, k in zip(val_rgb, val_y)])
print("遮罩中缺損像素的比例（訓練集）：", {LABELS_ZH[c]: f"{train_mask[train_y == k].mean():.1%}"
                                for k, c in enumerate(CATEGORIES)})

fig, axes = plt.subplots(2, len(CATEGORIES), figsize=(9, 6))
for j, (k, cat) in enumerate(enumerate(CATEGORIES)):
    i = np.where(train_y == k)[0][0]
    axes[0, j].imshow(train_rgb[i]); axes[0, j].set_title(cat)
    axes[1, j].imshow(train_mask[i], cmap="gray"); axes[1, j].set_title("mask")
for ax in axes.ravel(): ax.axis("off")
plt.show()

In [ ]:
def make_seg_ds(gray, mask, training):
    ds = tf.data.Dataset.from_tensor_slices((gray[..., None], mask[..., None]))   # 先以 uint8 存放，節省記憶體
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, tf.cast(y, tf.float32)),
                num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        def augment(img, m):
            both = tf.concat([img, m], axis=-1)          # 影像與遮罩做同樣的翻轉與旋轉
            both = tf.image.random_flip_left_right(both)
            both = tf.image.random_flip_up_down(both)
            both = tf.image.rot90(both, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
            return both[..., :1], both[..., 1:]
        ds = ds.shuffle(len(gray), seed=SEED).map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(UNET_BATCH).prefetch(tf.data.AUTOTUNE)

def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, activation="relu", padding="same")(x)
    return layers.Conv2D(filters, 3, activation="relu", padding="same")(x)

def unet(input_size=(IMG_SIZE, IMG_SIZE, 1)):
    inputs = keras.Input(input_size)
    skips, x = [], inputs
    for f in (64, 128, 256, 512):                         # 編碼：擷取特徵
        x = conv_block(x, f); skips.append(x)
        x = layers.MaxPooling2D(2)(x)
    x = conv_block(x, 1024)
    for f, skip in zip((512, 256, 128, 64), reversed(skips)):   # 解碼：上採樣並拼接還原尺寸
        x = layers.UpSampling2D(2)(x)
        x = conv_block(layers.concatenate([skip, x]), f)
    outputs = layers.Conv2D(1, 1, activation="sigmoid")(x)
    return keras.Model(inputs, outputs, name="unet")

def bce_dice_loss(y_true, y_pred):
    """binary cross-entropy＋Dice：缺損只占小部分時，Dice 讓模型不會全部猜背景。"""
    bce = tf.reduce_mean(keras.losses.binary_crossentropy(y_true, y_pred))
    inter = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
    total = tf.reduce_sum(y_true + y_pred, axis=[1, 2, 3])
    return bce + tf.reduce_mean(1.0 - (2.0 * inter + 1.0) / (total + 1.0))

seg_model = unet()
seg_model.compile(optimizer=keras.optimizers.Adam(1e-4), loss=bce_dice_loss,
                  metrics=["accuracy", keras.metrics.BinaryIoU(target_class_ids=[1], threshold=0.5, name="iou")])
print(f"U-Net 參數量：{seg_model.count_params():,}")

seg_history = seg_model.fit(
    make_seg_ds(train_gray, train_mask, True),
    validation_data=make_seg_ds(val_gray, val_mask, False),
    epochs=UNET_EPOCHS,
    callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
               keras.callbacks.ReduceLROnPlateau(factor=0.1, patience=2, min_lr=1e-7)])

In [ ]:
seg_loss, seg_acc, seg_iou = seg_model.evaluate(make_seg_ds(val_gray, val_mask, False), verbose=0)
print(f"U-Net 驗證集：像素準確率 {seg_acc:.2%}、缺損 IoU {seg_iou:.3f}、損失 {seg_loss:.3f}")
print("（比對的是自動產生的近似遮罩，不是人工標註）")

pred = seg_model.predict(val_gray[:3, ..., None] / 255.0, verbose=0)[..., 0]
fig, axes = plt.subplots(3, 3, figsize=(8, 8))
for i in range(3):
    for ax, img, title in zip(axes[i], (val_rgb[i], val_mask[i], pred[i] >= 0.5),
                              (CATEGORIES[val_y[i]], "pseudo mask", "U-Net prediction")):  # Colab 沒有中文字型，圖上用英文
        ax.imshow(img, cmap=None if img.ndim == 3 else "gray"); ax.set_title(title); ax.axis("off")
plt.show()

## 5. 整合：種類＋範圍＋面積＋建議

一張照片依序經過兩個模型：ResNet50V2 判斷種類，U-Net 標出範圍；再依範圍估計面積，最後交給 GPT 產生白話的處理建議。

In [ ]:
SYSTEM_PROMPT = (
    "你是協助一般住戶理解牆面問題的助理。請用繁體中文、白話、條列回答：\n"
    "1. 這可能是什麼問題、常見成因\n"
    "2. 嚴重程度，以及是否需要找專業人員\n"
    "3. 住戶可以先做的處理\n"
    "最後提醒：這只是 AI 的初步判斷，涉及結構安全時應請專業技師到場評估。")

def get_advice(label_zh, confidence, coverage, area_mm2):
    if client is None:
        return "（未設定 OPENAI_API_KEY，略過 AI 建議）"
    summary = (f"辨識結果：{label_zh}（信心 {confidence:.0%}）；缺損約占照片 {coverage:.0%}，"
               f"粗估面積 {area_mm2:,.0f} mm²（以每像素 {PIXEL_MM} mm 估算）。")
    try:
        response = client.chat.completions.create(
            model=OPENAI_MODEL, temperature=0.5, max_tokens=400,
            messages=[{"role": "system", "content": SYSTEM_PROMPT},
                      {"role": "user", "content": summary}])
        return response.choices[0].message.content
    except Exception as e:
        return f"AI 建議產生失敗：{e}"

def analyze(image):
    """輸入 RGB 照片，回傳種類機率、範圍疊圖、面積說明與建議。"""
    if image is None:
        raise gr.Error("請先上傳一張牆面照片")
    image = np.asarray(image)[..., :3].astype(np.uint8)
    rgb = cv2.resize(image, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)

    # 1) 種類
    probs = cls_model.predict(preprocess_input(rgb[None].astype("float32")), verbose=0)[0]
    k = int(probs.argmax())
    label_zh = LABELS_ZH[CATEGORIES[k]] if probs[k] >= MIN_CONFIDENCE else "無法確定"

    # 2) 範圍
    prob_map = seg_model.predict(to_gray(rgb)[None, ..., None] / 255.0, verbose=0)[0, ..., 0]
    mask = (prob_map >= 0.5).astype(np.uint8)

    # 3) 面積
    coverage = float(mask.mean())
    area_mm2 = float(mask.sum()) * PIXEL_MM ** 2

    # 4) 疊圖：把範圍以紅色標在原始照片上
    big = cv2.resize(mask, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST).astype(bool)
    overlay = image.copy()
    overlay[big] = (0.5 * overlay[big] + 0.5 * np.array([255, 0, 0])).astype(np.uint8)

    area_text = f"缺損約占照片 {coverage:.1%}\n粗估面積 {area_mm2:,.0f} mm²（假設每像素 {PIXEL_MM} mm）"
    advice = get_advice(label_zh, float(probs[k]), coverage, area_mm2)
    return {LABELS_ZH[c]: float(p) for c, p in zip(CATEGORIES, probs)}, overlay, area_text, advice

In [ ]:
# 用驗證集每類一張照片試跑整條流程
for k, cat in enumerate(CATEGORIES):
    i = int(np.where(val_y == k)[0][0])
    probs, overlay, area_text, advice = analyze(read_rgb(val_p[i], size=448))
    top = max(probs, key=probs.get)
    print(f"實際：{LABELS_ZH[cat]}｜預測：{top}（{probs[top]:.0%}）｜{area_text.splitlines()[0]}")
    plt.figure(figsize=(3, 3)); plt.imshow(overlay); plt.axis("off"); plt.show()
print("建議範例：\n", advice)

## 6. 互動介面

上傳牆面照片，一次看到種類、範圍、面積與建議。

In [ ]:
examples = [val_p[int(np.where(val_y == k)[0][0])] for k in range(len(CATEGORIES))]
demo = gr.Interface(
    fn=analyze,
    inputs=gr.Image(type="numpy", label="牆面照片"),
    outputs=[gr.Label(num_top_classes=len(CATEGORIES), label="種類（ResNet50V2）"),
             gr.Image(label="缺損範圍（U-Net）"),
             gr.Textbox(label="面積估計"),
             gr.Textbox(label="AI 處理建議", lines=10)],
    title="WallGuard 牆面危害辨識（完整版）",
    description="上傳一張室內牆面照片，辨識裂縫、壁癌或鏽蝕，標出範圍並給出處理建議。結果僅供初步參考。",
    examples=examples)
demo.launch(debug=True)

## 7. 儲存模型（可選）

模型檔很大（U-Net 約 120 MB），沒有放進 GitHub。需要保留時，執行下面這格後從左側「檔案」下載，或複製到自己的 Google 雲端硬碟。

In [ ]:
model_dir = os.path.join(WORK_DIR, "models")
os.makedirs(model_dir, exist_ok=True)
cls_model.save(os.path.join(model_dir, "wallguard_resnet50v2.keras"))
seg_model.save(os.path.join(model_dir, "wallguard_unet.keras"))
print("已儲存到", model_dir)